# Tutorial 16: Hyperdimensional Q-Learning (QHD) with Gridworlds

This tutorial implements **QHD** — a brain-inspired, off-policy reinforcement learning algorithm that replaces the deep neural network in DQN with a Hyperdimensional Computing (HDC) model.

Based on: *"Efficient Off-Policy Reinforcement Learning via Brain-Inspired Computing"* (Ni et al., GLSVLSI 2023)

## What You'll Learn

- How to encode continuous states using **Fractional Power Encoding** (FPE)
- How to represent the Q-function as **one hypervector per action** (no neural network!)
- How to implement a **Hebbian update rule** that replaces backpropagation
- How to train QHD on two gridworld environments
- Why QHD works with a **batch size of just 2**

## Algorithm Overview

| Property | DQN | QHD |
|---|---|---|
| Q-function size | Millions of parameters | **D floats per action** |
| Training | Backpropagation | **Hebbian update** |
| Min batch size | 32–128 | **2** |
| Hardware | GPU | **Edge devices** |

### The Three Steps

1. **Encode** state $S=[s_1,\ldots,s_n]$ as: $\mathbf{S}_{hv} = \mathbf{p}_1^{s_1} \odot \cdots \odot \mathbf{p}_n^{s_n}$  
   where $\mathbf{p}_k^{s_k} = \exp(i\cdot\angle(\mathbf{p}_k)\cdot s_k)$ (fractional complex power)

2. **Predict** Q-value: $q(s,A) = \operatorname{Re}(\mathbf{M}_A \cdot \overline{\mathbf{S}_{hv}}) / D$

3. **Update**: $\mathbf{M}_A \leftarrow \mathbf{M}_A + \beta\cdot(q_{\text{true}} - q_{\text{pred}})\cdot\mathbf{S}_{hv}$

## Setup

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from collections import deque
import random

from vsax import create_fhrr_model
from vsax.sampling import sample_complex_random
from vsax.similarity import cosine_similarity

# Reproducibility
np.random.seed(42)
random.seed(42)

# Create FHRR model (paper uses D=6000; D=2000 is sufficient for gridworld)
model = create_fhrr_model(dim=2000)
print(f"VSA Model: {model.rep_cls.__name__}")
print(f"Dimension: D = {model.dim}")
print(f"Each action's Q-function is stored in a single D-dimensional hypervector")

## Section 1: QHD State Encoder — Fractional Power Encoding

The key insight: **similar states produce similar hypervectors** because fractional power is continuous.  
State $(0.3, 0.7)$ is close to $(0.31, 0.7)$ in hypervector space.

Each state dimension $s_k$ is encoded as $\mathbf{p}_k^{s_k} = \exp(i \cdot \angle(\mathbf{p}_k) \cdot s_k)$ where $\mathbf{p}_k$ is a random complex hypervector.

In [ ]:
class QHDStateEncoder:
    """Fractional Power Encoding for continuous state vectors.

    For a state S = [s_1, s_2, ..., s_n], encodes as:
        S_hv = p_1^{s_1} * p_2^{s_2} * ... * p_n^{s_n}

    where p_k is a random base complex hypervector, * is elementwise multiply
    (FHRR bind), and ^ is fractional complex exponentiation:
        p_k^{s_k} = exp(i * angle(p_k) * s_k)
    """

    def __init__(self, model, n_dims, seed=42):
        key = jax.random.PRNGKey(seed)
        keys = jax.random.split(key, n_dims)
        # One random base HV per state dimension
        self.bases = [sample_complex_random(model.dim, 1, keys[i])[0] for i in range(n_dims)]
        self.dim = model.dim
        self.n_dims = n_dims

    def encode(self, state):
        """Encode a state vector as a complex hypervector."""
        hv = jnp.ones(self.dim, dtype=jnp.complex64)
        for k, sk in enumerate(state):
            angles = jnp.angle(self.bases[k])
            powered = jnp.exp(1j * angles * float(sk)).astype(jnp.complex64)
            hv = hv * powered
        return hv

In [ ]:
# Verify: different states are quasi-orthogonal
encoder = QHDStateEncoder(model, n_dims=2)

states = [(0.0, 0.0), (0.333, 0.333), (0.667, 0.667), (1.0, 1.0)]
hvs = [encoder.encode(s) for s in states]

print("Cosine similarity between state encodings:")
print("(Random hypervectors have ~0 similarity — this confirms encoding works)")
print()
for i in range(len(states)):
    for j in range(i + 1, len(states)):
        sim = cosine_similarity(hvs[i], hvs[j])
        print(f"  sim({states[i]}, {states[j]}) = {sim:.4f}")

## Section 2: QHD Agent

The agent stores **one complex hypervector per action** (initialized to zeros). The Q-value is the real part of the inner product, normalized by D.

The update `M_A += lr * error * S_hv` is a **Hebbian learning rule** — the model vector accumulates weighted state hypervectors, just like synaptic strengthening in biological neural networks.

In [ ]:
class QHDAgent:
    """QHD off-policy RL agent using Hyperdimensional Computing.

    Q-function: one model hypervector M_A per action.
    Q-value:    q(s, a) = Re(M_A . conj(S_hv)) / D
    Update:     M_A += lr * (q_true - q_pred) * S_hv
    """

    def __init__(self, model, n_state_dims, n_actions,
                 lr=0.1, gamma=0.95,
                 eps_start=1.0, eps_end=0.05, eps_decay=0.99,
                 buffer_size=200, seed=42):
        self.n_actions = n_actions
        self.lr = lr
        self.gamma = gamma
        self.eps = eps_start
        self.eps_end = eps_end
        self.eps_decay = eps_decay
        self.dim = model.dim

        # One model HV per action, initialized to zeros
        self.models = [jnp.zeros(model.dim, dtype=jnp.complex64) for _ in range(n_actions)]

        # State encoder
        self.encoder = QHDStateEncoder(model, n_state_dims, seed=seed)

        # Experience replay buffer
        self.buffer = deque(maxlen=buffer_size)

    def q_value(self, state_hv, action):
        """Compute Q-value: Re(M_A . conj(S_hv)) / D."""
        return float(jnp.real(jnp.dot(self.models[action], jnp.conj(state_hv))) / self.dim)

    def update(self, state_hv, action, error):
        """Hebbian update: M_A += lr * error * S_hv."""
        self.models[action] = self.models[action] + self.lr * error * state_hv

    def act(self, state):
        """Epsilon-greedy action selection."""
        if random.random() < self.eps:
            return random.randint(0, self.n_actions - 1)
        state_hv = self.encoder.encode(state)
        q_values = [self.q_value(state_hv, a) for a in range(self.n_actions)]
        return int(np.argmax(q_values))

    def remember(self, state, action, reward, next_state, done):
        """Store transition in experience replay buffer."""
        self.buffer.append((state, action, reward, next_state, done))

    def train_step(self, batch):
        """Bellman + Hebbian update on a sampled batch."""
        for state, action, reward, next_state, done in batch:
            state_hv = self.encoder.encode(state)
            q_pred = self.q_value(state_hv, action)

            if done:
                q_true = reward
            else:
                next_hv = self.encoder.encode(next_state)
                q_next = max(self.q_value(next_hv, a) for a in range(self.n_actions))
                q_true = reward + self.gamma * q_next

            error = q_true - q_pred
            self.update(state_hv, action, error)

    def decay_epsilon(self):
        """Exponential epsilon decay."""
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

## Section 3: 4×4 GridWorld

```
S . . .
. . . .
. . . .
. . . G
```

- **State**: `[row/3, col/3]` normalized to [0, 1]
- **Actions**: Up (↑), Down (↓), Left (←), Right (→)
- **Rewards**: +1.0 at goal, −0.01 per step
- **Max steps**: 100

In [ ]:
class GridWorld4x4:
    """Simple 4x4 grid: Start at (0,0), Goal at (3,3)."""

    SIZE = 4
    GOAL = (3, 3)
    MAX_STEPS = 100
    ACTION_DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right
    ACTION_ARROWS = ['\u2191', '\u2193', '\u2190', '\u2192']

    def __init__(self):
        self.state = None
        self.steps = 0

    def reset(self):
        self.state = (0, 0)
        self.steps = 0
        return self._normalize(self.state)

    def _normalize(self, state):
        r, c = state
        return [r / (self.SIZE - 1), c / (self.SIZE - 1)]

    def step(self, action):
        r, c = self.state
        dr, dc = self.ACTION_DELTAS[action]
        nr = max(0, min(self.SIZE - 1, r + dr))
        nc = max(0, min(self.SIZE - 1, c + dc))
        self.state = (nr, nc)
        self.steps += 1

        if self.state == self.GOAL:
            return self._normalize(self.state), 1.0, True
        elif self.steps >= self.MAX_STEPS:
            return self._normalize(self.state), -0.01, True
        else:
            return self._normalize(self.state), -0.01, False

In [ ]:
def train(env, agent, n_episodes, batch_size=4):
    """Train agent using QHD with experience replay."""
    rewards_history = []
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False
        while not done:
            action = agent.act(state)
            next_state, reward, done = env.step(action)
            agent.remember(state, action, reward, next_state, done)
            if len(agent.buffer) >= batch_size:
                batch = random.sample(list(agent.buffer), batch_size)
                agent.train_step(batch)
            state = next_state
            total_reward += reward
        agent.decay_epsilon()
        rewards_history.append(total_reward)
    return rewards_history


# Train on 4x4
env4 = GridWorld4x4()
agent4 = QHDAgent(model, n_state_dims=2, n_actions=4,
                  lr=0.1, gamma=0.95, eps_start=1.0, eps_end=0.05,
                  eps_decay=0.99, buffer_size=200)

print("Training QHD on 4x4 GridWorld...")
rewards4 = train(env4, agent4, n_episodes=500, batch_size=4)
print(f"Final 50-episode average reward: {np.mean(rewards4[-50:]):.3f}")
print(f"Final epsilon: {agent4.eps:.4f}")

In [ ]:
def plot_learning_curve(rewards, window=30, title="QHD Learning Curve"):
    """Plot reward vs episode with rolling average."""
    rolling = [np.mean(rewards[max(0, i - window):i + 1]) for i in range(len(rewards))]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(rewards, alpha=0.25, color='steelblue', label='Episode reward')
    ax.plot(rolling, color='steelblue', linewidth=2.5, label=f'{window}-ep rolling avg')
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Episode', fontsize=12)
    ax.set_ylabel('Total Reward', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return rolling


rolling4 = plot_learning_curve(rewards4, title="QHD on 4x4 GridWorld (batch_size=4)")

In [ ]:
def visualize_policy(agent, env, title="Learned Policy"):
    """Visualize greedy policy as arrow grid and Q-value heatmap."""
    size = env.SIZE
    obstacles = getattr(env, 'OBSTACLES', frozenset())

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # --- Policy Grid ---
    ax = axes[0]
    ax.set_xlim(-0.5, size - 0.5)
    ax.set_ylim(-0.5, size - 0.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13)

    q_grid = np.full((size, size), np.nan)

    for r in range(size):
        for c in range(size):
            row_plot = size - 1 - r  # flip y-axis for display

            if (r, c) in obstacles:
                rect = plt.Rectangle((c - 0.5, row_plot - 0.5), 1, 1, color='#555555')
                ax.add_patch(rect)
                ax.text(c, row_plot, '\u25a0', ha='center', va='center',
                        fontsize=18, color='white')
                continue

            if (r, c) == env.GOAL:
                rect = plt.Rectangle((c - 0.5, row_plot - 0.5), 1, 1,
                                     color='#90EE90', alpha=0.7)
                ax.add_patch(rect)
                ax.text(c, row_plot, 'G', ha='center', va='center',
                        fontsize=16, color='darkgreen', fontweight='bold')
                q_grid[r, c] = 1.0
                continue

            if (r, c) == (0, 0):
                rect = plt.Rectangle((c - 0.5, row_plot - 0.5), 1, 1,
                                     color='#ADD8E6', alpha=0.5)
                ax.add_patch(rect)

            # Compute best action
            state = env._normalize((r, c))
            state_hv = agent.encoder.encode(state)
            q_vals = [agent.q_value(state_hv, a) for a in range(agent.n_actions)]
            best_a = int(np.argmax(q_vals))
            q_grid[r, c] = max(q_vals)

            label = 'S' if (r, c) == (0, 0) else env.ACTION_ARROWS[best_a]
            ax.text(c, row_plot, label, ha='center', va='center',
                    fontsize=18, color='#222222')

    for i in range(size + 1):
        ax.axhline(i - 0.5, color='black', linewidth=0.8)
        ax.axvline(i - 0.5, color='black', linewidth=0.8)

    ax.set_xticks(range(size))
    ax.set_yticks(range(size))
    ax.set_xticklabels([f'c{i}' for i in range(size)])
    ax.set_yticklabels([f'r{i}' for i in range(size - 1, -1, -1)])
    ax.tick_params(labelsize=9)

    # --- Q-value Heatmap ---
    ax2 = axes[1]
    masked = np.ma.array(q_grid, mask=np.isnan(q_grid))
    im = ax2.imshow(masked, cmap='RdYlGn', vmin=-0.5, vmax=1.0, origin='upper')
    plt.colorbar(im, ax=ax2, label='Max Q-value', fraction=0.046, pad=0.04)
    ax2.text(env.GOAL[1], env.GOAL[0], 'G', ha='center', va='center',
             fontsize=12, color='white', fontweight='bold')
    ax2.text(0, 0, 'S', ha='center', va='center', fontsize=12, color='white')
    ax2.set_title('Max Q-value Heatmap', fontsize=13)
    ax2.set_xlabel('Column')
    ax2.set_ylabel('Row')
    ax2.set_xticks(range(size))
    ax2.set_yticks(range(size))

    plt.tight_layout()
    plt.show()


visualize_policy(agent4, env4, title="Learned Policy \u2014 4\u00d74 GridWorld")

**What to look for:**
- Arrows should point generally toward the goal (bottom-right)
- Q-value heatmap should be brightest (highest) near the goal
- The learned policy navigates the agent from S to G efficiently

## Section 4: 5×5 GridWorld with Obstacles

```
S . . . .
. X . X .
. . . . .
. X . X .
. . . . G
```

- **X** = impassable walls (agent bounces back, receives standard −0.01 step penalty)
- **State**: `[row/4, col/4]` normalized to [0, 1]
- **Max steps**: 200

The agent must learn to navigate around walls — a harder credit assignment problem.

In [ ]:
class GridWorld5x5:
    """5x5 grid with obstacles. Start at (0,0), Goal at (4,4).

    X = impassable wall (agent stays in place, normal step penalty).
    """

    SIZE = 5
    GOAL = (4, 4)
    OBSTACLES = frozenset({(1, 1), (1, 3), (3, 1), (3, 3)})
    MAX_STEPS = 200
    ACTION_DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    ACTION_ARROWS = ['\u2191', '\u2193', '\u2190', '\u2192']

    def __init__(self):
        self.state = None
        self.steps = 0

    def reset(self):
        self.state = (0, 0)
        self.steps = 0
        return self._normalize(self.state)

    def _normalize(self, state):
        r, c = state
        return [r / (self.SIZE - 1), c / (self.SIZE - 1)]

    def step(self, action):
        r, c = self.state
        dr, dc = self.ACTION_DELTAS[action]
        nr = max(0, min(self.SIZE - 1, r + dr))
        nc = max(0, min(self.SIZE - 1, c + dc))

        # Bounce back from obstacles (impassable walls)
        if (nr, nc) in self.OBSTACLES:
            nr, nc = r, c

        self.state = (nr, nc)
        self.steps += 1

        if self.state == self.GOAL:
            return self._normalize(self.state), 1.0, True
        elif self.steps >= self.MAX_STEPS:
            return self._normalize(self.state), -0.01, True
        else:
            return self._normalize(self.state), -0.01, False

In [ ]:
env5 = GridWorld5x5()
agent5 = QHDAgent(model, n_state_dims=2, n_actions=4,
                  lr=0.1, gamma=0.95, eps_start=1.0, eps_end=0.05,
                  eps_decay=0.99, buffer_size=500)

print("Training QHD on 5x5 GridWorld with obstacles...")
rewards5 = train(env5, agent5, n_episodes=1000, batch_size=4)
print(f"Final 50-episode average reward: {np.mean(rewards5[-50:]):.3f}")

In [ ]:
rolling5 = plot_learning_curve(rewards5, title="QHD on 5x5 GridWorld with Obstacles (batch_size=4)")
visualize_policy(agent5, env5, title="Learned Policy \u2014 5\u00d75 GridWorld")

## Section 5: Effect of Batch Size

A key claim from Ni et al. (2023): QHD works well with **batch size as small as 2**, while DQN typically requires 32–128. This dramatically reduces memory bandwidth requirements on embedded hardware.

Let's verify this experimentally.

In [ ]:
def run_batch_size_experiment(env_cls, model, batch_sizes, n_episodes=400, n_runs=3):
    """Compare QHD performance across different batch sizes.

    Returns dict: batch_size -> array of shape (n_runs, n_episodes)
    """
    results = {}
    for bs in batch_sizes:
        print(f"  batch_size={bs}...", end=" ", flush=True)
        all_runs = []
        for run in range(n_runs):
            env = env_cls()
            agent = QHDAgent(
                model, n_state_dims=2, n_actions=4,
                lr=0.1, gamma=0.95,
                eps_start=1.0, eps_end=0.05, eps_decay=0.99,
                buffer_size=200, seed=run * 10 + 42
            )
            # Reset global seeds per run for reproducibility
            np.random.seed(run * 100)
            random.seed(run * 100)
            rewards = train(env, agent, n_episodes=n_episodes, batch_size=bs)
            all_runs.append(rewards)
        results[bs] = np.array(all_runs)
        mean_final = np.mean(results[bs][:, -50:])
        print(f"mean final reward: {mean_final:.3f}")
    return results


print("Running batch size experiment on 4x4 GridWorld...")
batch_sizes = [2, 4, 8, 16]
bs_results = run_batch_size_experiment(
    GridWorld4x4, model, batch_sizes, n_episodes=400, n_runs=3
)

In [ ]:
window = 30
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

fig, ax = plt.subplots(figsize=(11, 5))

for bs, color in zip(batch_sizes, colors):
    runs = bs_results[bs]  # shape (n_runs, n_episodes)
    mean_r = np.mean(runs, axis=0)
    std_r = np.std(runs, axis=0)
    rolling = np.array([np.mean(mean_r[max(0, i - window):i + 1]) for i in range(len(mean_r))])

    ax.fill_between(
        range(len(rolling)),
        rolling - std_r,
        rolling + std_r,
        alpha=0.12, color=color
    )
    ax.plot(rolling, color=color, linewidth=2.2, label=f'batch_size={bs}')

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel(f'Total Reward ({window}-ep rolling avg)', fontsize=12)
ax.set_title(
    'QHD Performance vs. Batch Size\n(4\u00d74 GridWorld, 3 runs averaged)',
    fontsize=14
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary table
print(f"\nFinal 50-episode average reward by batch size:")
print(f"{'Batch Size':>12} | {'Mean Reward':>12} | {'Std':>8}")
print("-" * 38)
for bs in batch_sizes:
    final = bs_results[bs][:, -50:].mean(axis=1)
    print(f"{bs:>12} | {np.mean(final):>12.4f} | {np.std(final):>8.4f}")

## Key Takeaways

1. **One HV per action** replaces an entire Q-network — the complete Q-function for 4 actions with D=2000 uses ~128 KB

2. **Hebbian update** (`M_A += lr * error * S_hv`) replaces backpropagation — no gradients, no chain rule

3. **Fractional Power Encoding** maps continuous states to quasi-orthogonal complex hypervectors; nearby states have nearby encodings

4. **Batch size 2 works** — QHD's inner product update is efficient enough that a tiny replay buffer suffices

5. **Naturally GPU-compatible** — the state encoding and Q-value computation are all JAX operations

6. **Brain-inspired** — the update rule mirrors synaptic strengthening: states paired with high rewards strengthen the model HV in the direction of the state HV

## Next Steps

- Scale to CartPole (4D continuous state space)
- Add `@jax.jit` to `encode()` and `q_value()` for faster inference
- Compare training time vs DQN on the same environment
- Explore multi-step returns and prioritized experience replay
- See Tutorial 11 for more Fractional Power Encoding examples
- See Tutorial 6 for VSA vs neural network efficiency comparisons